# Phase 2: predict neuron sensitivity across networks

This notebook reviews five independently trained MNIST MLPs and asks whether properties available before ablation predict the fraction of test predictions changed when a neuron is removed. Training seed is the replication unit; each validation fold holds out a complete network.

## 1. Load the recorded run

The full run and all five checkpoints are saved under `results/phase2/`. These cells inspect recorded data; they do not retrain models.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Image

ROOT = Path('results/phase2')
metrics = pd.read_csv(ROOT / 'baseline_metrics.csv')
training = pd.read_csv(ROOT / 'training_metrics.csv')
neurons = pd.read_csv(ROOT / 'neuron_properties_and_ablation.csv')
correlations = pd.read_csv(ROOT / 'feature_correlations_by_model.csv')
cv = pd.read_csv(ROOT / 'heldout_model_predictive_performance.csv')
predictions = pd.read_csv(ROOT / 'heldout_model_predictions.csv')
print('models:', metrics.seed.nunique(), 'neuron records:', len(neurons), 'test images/model: 10,000')

## 2. Baseline performance and sensitivity target

Each model was trained for five epochs with Adam and cross-entropy. The primary target is `prediction_flip_rate`: the number of test examples whose predicted class changed after ablation, divided by 10,000. Accuracy can hide offsetting flips from correct to wrong and wrong to correct, so flip rate is measured separately.

In [ ]:
display(metrics[['seed','baseline_accuracy','baseline_correct','train_loss_final']])
print('Mean accuracy:', f"{metrics.baseline_accuracy.mean():.3%}")
print('SD across model seeds:', f"{metrics.baseline_accuracy.std(ddof=1)*100:.3f} percentage points")
display(neurons.groupby('layer').prediction_flip_pct.agg(['mean','median','max']))

## 3. Why the local logit feature has a ReLU gate

Let `h1 = ReLU(W1 x + b1)`, `h2 = ReLU(W2 h1 + b2)`, and `z = W3 h2 + b3`. For first-layer neuron j, the local contribution estimate is `h1[j] · ∂z/∂h1[j]`. The derivative includes the second ReLU gate: `∂z_c/∂h1_j = Σ_k W3[c,k] · 1[(W2 h1+b2)_k > 0] · W2[k,j]`. This is a first-order local approximation, not the exact finite effect of removing the unit. The ablation flip target comes from the exact modified network forward pass. For a second-layer unit i, `h2[i] W3[:,i]` is its exact additive baseline logit contribution because the final layer is linear.

Incoming/outgoing norms need only the parameters. Activation frequency, mean activation, and local logit contribution use baseline forward passes over the test set. Prediction flips and margin changes require a separate ablation evaluation.

In [ ]:
summary = pd.read_csv(ROOT / 'feature_correlation_summary.csv')
display(summary.pivot(index='feature', columns='layer', values='mean_rho'))
print('Feature correlations are summarized across five independently trained models; SDs are between-model SDs.')

## 4. Hold out entire networks

Each fold trains on the neuron rows from four seeds and evaluates on all neurons from the fifth seed. Standardization is fitted only on the four training models. The table compares incoming norm alone against all five pre-ablation properties.

In [ ]:
cv_summary = pd.read_csv(ROOT / 'heldout_model_predictive_summary.csv')
display(cv_summary)
assert cv.n_train_models.eq(4).all()
assert cv.n_test_neurons.isin([64,128]).all()
print('The held-out seed is excluded from each predictor fit.')

## 5. Do neurons 104 and 122 repeat across seeds?

The numeric index is not a shared functional identity across random initializations. This table is a check on those positions only; it cannot establish that the same learned feature recurs.

In [ ]:
index_check = pd.read_csv(ROOT / 'phase1_index_comparison.csv')
display(index_check.pivot_table(index='seed', columns='neuron', values='prediction_flip_pct'))
display(pd.read_csv(ROOT / 'matched_norm_sensitivity_spread.csv'))

## 6. Inspect figures

The SVG figures are vector originals; PNG previews are provided for easy viewing.

In [ ]:
for name in ['phase2_sensitivity_heatmaps.png', 'feature_spearman_comparison.png', 'heldout_model_predicted_vs_observed.png', 'layerwise_distributions_by_seed.png', 'selected_neuron_margin_changes.png']:
    print(name)
    display(Image(filename=str(ROOT / 'figures' / name)))

## Interpretation and limits

Combined pre-ablation properties improved held-out-model ranking and error over incoming norm alone in both layers. The result is promising for this small MLP and MNIST, but it is not a general importance theorem. There are only five model seeds, and the validation holds out models on the same fixed image distribution, not datasets. See `PHASE2_REPORT.md` for all fold values, measurement details, and limitations.